In [ ]:
# CYR-GPU-006 — CELL 0: FREEZE / TEST / GPU CALIBRATION / RESOLVE
import hashlib, importlib.util, json, os, subprocess, sys
from pathlib import Path

REPO = Path("/content/An-Ra-the-new-AGI")
BRANCH = "cymek-500m-readiness"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# The repo needs tokenizers; pytest is used for a fast deterministic preflight.
missing = [name for name in ("tokenizers", "pytest") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

PREREG_PATH = REPO / "docs/cymek/experiments/CYR-GPU-006/PREREGISTRATION.json"
PREREG = json.loads(PREREG_PATH.read_text("utf-8"))
EXTERNAL = Path("/content/CYR-GPU-006-PREREGISTRATION.json")
EXTERNAL.write_text(json.dumps(PREREG, indent=2), encoding="utf-8")
EXECUTABLE_SHA = PREREG["executable_sha"]
subprocess.run(["git", "checkout", "-q", EXECUTABLE_SHA], check=True)
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], check=True,
                      capture_output=True, text=True).stdout.strip()
assert HEAD == EXECUTABLE_SHA, (HEAD, EXECUTABLE_SHA)
for relative, expected in PREREG["executable_files"].items():
    actual = hashlib.sha256((REPO / relative).read_bytes()).hexdigest()
    assert actual == expected, f"hash mismatch: {relative}"
for relative, expected_blob in PREREG.get("dependency_blobs", {}).items():
    actual_blob = subprocess.run(["git", "hash-object", relative], check=True,
                                 capture_output=True, text=True).stdout.strip()
    assert actual_blob == expected_blob, f"dependency blob mismatch: {relative}"
print("frozen executable verified", HEAD)

# Cheap exact-code preflight after checkout. Any failure blocks GPU science.
subprocess.run([sys.executable, "-m", "py_compile",
                "v5_experiments/cyr_gpu006.py", "anra_v5/cyr_gpu006_run.py"], check=True)
subprocess.run([sys.executable, "-m", "pytest",
                "tests/test_v5_cyr_gpu006_core.py",
                "tests/test_v5_cyr_gpu006_notebook.py", "-q"], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CYR-GPU-006 requires a Google Colab CUDA GPU")
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

from anra_v5.cyr_gpu006_run import production_tokenizer, calibrate_candidates
from v5_experiments import cyr_gpu006 as core

tokenizer, identity = production_tokenizer(REPO)
assert identity["artifact_sha256"] == PREREG["tokenizer"]["artifact_sha256"]
splits = core.render_t2_worlds()
manifest = core.build_data_manifest(splits)
core.assert_manifest_sha(manifest)
assert manifest["sha256"] == PREREG["data"]["data_manifest_sha256_full"]
audit = core.commutation_audit(splits, tv_bound=0.20)
assert audit["commutation_free"], audit["findings"]
registry = core.proxy_registry(vocab_size=identity["vocabulary_size"])
special = {"pad_id": identity["pad_id"], "bos_id": identity["bos_id"], "eos_id": identity["eos_id"]}
CALIBRATIONS = calibrate_candidates(registry=registry, tokenizer=tokenizer,
                                    torch=torch, device=DEVICE, special=special,
                                    train_rows=splits["train"],
                                    eval_rows=splits["dev_controller"])
RESOLVED = core.resolve_from_calibrations(CALIBRATIONS)
core.validate_resolved(RESOLVED)
Path("/content/CYR-GPU-006-CALIBRATIONS.json").write_text(json.dumps(CALIBRATIONS, indent=2))
Path("/content/CYR-GPU-006-RESOLVED.json").write_text(json.dumps(RESOLVED, indent=2))
print(json.dumps(RESOLVED, indent=2))
print("CYR-GPU-006 PREEXECUTION GATE: PASS")


In [ ]:
# CYR-GPU-006 — CELL 1: DURABLE RUN / STAGE-LEVEL RESUME
import json, sys
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/An-Ra-the-new-AGI")
sys.path.insert(0, str(REPO))
DEVICE = torch.device("cuda")
assert torch.cuda.is_available() and DEVICE.type == "cuda"
PREREG = json.loads(Path("/content/CYR-GPU-006-PREREGISTRATION.json").read_text())
CALIBRATIONS = json.loads(Path("/content/CYR-GPU-006-CALIBRATIONS.json").read_text())
RESOLVED = json.loads(Path("/content/CYR-GPU-006-RESOLVED.json").read_text())
OUT = Path("/content/drive/MyDrive/CYMEK/CYR-GPU-006")
OUT.mkdir(parents=True, exist_ok=True)

from anra_v5.cyr_gpu006_run import run_campaign
try:
    campaign = run_campaign(repo=REPO, out=OUT, preregistration=PREREG,
                            resolved=RESOLVED, calibrations=CALIBRATIONS,
                            torch=torch, device=DEVICE,
                            progress=lambda msg: print(msg, flush=True))
    print("campaign:", campaign["status"], campaign["decision"]["verdict"],
          campaign["decision"].get("winner"))
except Exception as exc:
    print("RUN FAILED, but partial evidence was packaged on Drive:", repr(exc))
    print("OUT:", OUT)
    raise


In [ ]:
# CYR-GPU-006 — CELL 2: VERIFY / DOWNLOAD RESULT BUNDLE
import hashlib, json
from pathlib import Path
from google.colab import files

OUT = Path("/content/drive/MyDrive/CYMEK/CYR-GPU-006")
receipt = json.loads((OUT / "campaign_receipt.json").read_text("utf-8"))
bundle = Path(receipt["bundle"]["path"])
assert bundle.exists(), bundle
actual = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert actual == receipt["bundle"]["sha256"], "bundle hash mismatch"
print("verified", bundle.name, actual)
print("status:", receipt["status"], "verdict:", receipt.get("decision", {}).get("verdict"))
files.download(str(bundle))
